In [1]:
TOLERANCE = 1e-9
from typing import Dict, Tuple
import numpy as np
from classHMM import HMM

In [2]:
def forward(hmm, observations: str) -> Tuple[np.ndarray, float]:
    """Finds the total probability of the observed sequence across all paths given the HMM probabilities.

    Args:
        observations (str): observed symbols, i.e. "GGCACTGAA"

    Returns:
        Tuple[np.ndarray, float]: the forward trellis and the total log-probability

    Examples:
        >>> forward(hmm, 'G')
        (array([[-2.12026354],
                [-1.83258146]]), np.float64(-1.2729656758128873))
        >>> forward(hmm, 'GG')
        (array([[-2.12026354, -3.54737989],
                [-1.83258146, -2.91139113]]), np.float64(-2.486507931154974))
    """

    #Initialize matrix
    hmm.check_observations(observations)
    num_positions = len(observations)
    score_matrix = np.full((len(hmm.states), num_positions), -np.inf)
    #Get initial probability
    first_symbol = observations[0]

    for state in hmm.states:
         score_matrix[hmm.state_index[state], 0] = hmm.log_init[state] + hmm.log_emit[state][first_symbol]

    #Looping through observations and summing up scores
    for column in range(1, num_positions):
        symbol = observations[column]
        for current_state in hmm.states:
            incoming = []
            for previous_state in hmm.states:
                incoming.append(score_matrix[hmm.state_index[previous_state], column - 1]
                + hmm.log_trans[previous_state][current_state])
            score_matrix[hmm.state_index[current_state], column] = np.logaddexp.reduce(incoming) + hmm.log_emit[current_state][symbol]

    last_col_sum = np.logaddexp.reduce(score_matrix[:, -1])
    return score_matrix, last_col_sum


def backward(hmm, observations: str) -> Tuple[np.ndarray, float]:
    """Finds the total probability of the observed sequence across all paths given the HMM probabilities.

    Args:
        observations (str): observed symbols, i.e. "GGCACTGAA"

    Returns:
        Tuple[np.ndarray, float]: the backward trellis and the total log-probability

    Examples:
        >>> backward(hmm, 'G')
        (array([[0.],
                [0.]]), np.float64(-1.2729656758128873))
        >>> backward(hmm, 'GG')
        (array([[-1.42711636, 0.],
                [-1.07880966, 0.]]), np.float64(-2.486507931154974))
    """
    #Initialize Matrix
    hmm.check_observations(observations)
    num_positions = len(observations)
    score_matrix = np.full((len(hmm.states), num_positions), -np.inf)
    score_matrix[:,-1] = 0

    for column in range(num_positions - 2, -1, -1):
        next_symbol = observations[column + 1]
        for current_state in hmm.states:
            outgoing = []
            for next_state in hmm.states:
                outgoing.append(hmm.log_trans[current_state][next_state] + hmm.log_emit[next_state][next_symbol] + score_matrix[hmm
                                .state_index[next_state], column + 1])
            score_matrix[hmm.state_index[current_state], column] = np.logaddexp.reduce(outgoing)

    first_symbol = observations[0]
    finish_terms = []
    for state in hmm.states:
        finish_terms.append(hmm.log_init[state] + hmm.log_emit[state][first_symbol] + score_matrix[hmm.state_index[state], 0])
    log_prob = np.logaddexp.reduce(finish_terms)

    return score_matrix, log_prob


def forward_backward(hmm, observations: str) -> Dict[str, object]:
    """Calculates the posterior probability of each hidden state at each position given the entire observation sequence.

    Args:
        observations (str): observed symbols, i.e. "GGCACTGAA"

    Returns:
        Dict[str, object]: a dictionary containing:
            posterior: posterior probability matrix
            forward: forward log-probability matrix
            backward: backward log-probability matrix
            log_prob: total sequence log-probability
            
    Examples:
        >>> forward_backward(hmm, 'G')
        {'posterior': array([[0.42857143],
                            [0.57142857]]),
        'forward': array([[-2.12026354],
                        [-1.83258146]]),
        'backward': array([[0.],
                            [0.]]),
        'log_prob': np.float64(-1.2729656758128873)}
        >>> forward_backward(hmm, 'GG')
        {'posterior': array([[0.34615385, 0.34615385],
                            [0.65384615, 0.65384615]]),
        'forward': array([[-2.12026354, -3.54737989],
                        [-1.83258146, -2.91139113]]),
        'backward': array([[-1.42711636, 0.],
                            [-1.07880966, 0.]]),
        'log_prob': np.float64(-2.486507931154974)}
    """
    forward_matrix, forward_log_prob = forward(hmm, observations)
    backward_matrix, backward_log_prob = backward(hmm, observations)

    if not np.isclose(forward_log_prob, backward_log_prob, rtol=TOLERANCE, atol=0):
        raise ValueError("forward and backward log P(x) don't match")

    posterior_matrix = np.exp(forward_matrix + backward_matrix - forward_log_prob)

    if not np.allclose(posterior_matrix.sum(axis=0), 1, rtol=TOLERANCE, atol=0):
        raise ValueError("a posterior column doesn't sum to 1")

    return {"posterior": posterior_matrix, "forward": forward_matrix, "backward": backward_matrix, "log_prob": forward_log_prob}


In [3]:
# Example HMM from Project 09 instructions

obs = "ATGCAA"

init_probs = {
    "E": 0.6,
    "I": 0.4
}

trans_probs = {
    "E": {"E": 0.8, "I": 0.2},
    "I": {"E": 0.3, "I": 0.7}
}

emit_probs = {
    "E": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3},
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1}
}

hmm = HMM(init_probs, trans_probs, emit_probs)

In [4]:
# Test 1: Single observation
forward_matrix, forward_log_prob = forward(hmm, "A")
backward_matrix, backward_log_prob = backward(hmm, "A")
result = forward_backward(hmm, "A")

# Expected values calculated by hand
expected_log_prob = np.log(0.22)
expected_posterior = np.array([[0.18 / 0.22], [0.04 / 0.22]])

# Validate results
assert np.isclose(forward_log_prob, expected_log_prob, rtol=TOLERANCE, atol=0)
assert np.isclose(backward_log_prob, expected_log_prob, rtol=TOLERANCE, atol=0)
assert np.allclose(result["posterior"], expected_posterior, rtol=TOLERANCE, atol=0)

print("Test 1 passed!")

Test 1 passed!


In [5]:
# Test 2: Two observations
forward_matrix, forward_log_prob = forward(hmm, "GG")
backward_matrix, backward_log_prob = backward(hmm, "GG")
result = forward_backward(hmm, "GG")

# Expected values calculated by hand
expected_log_prob = np.log(0.0832)
expected_posterior = np.array([
    [(0.0192 + 0.0096) / 0.0832, (0.0192 + 0.0096) / 0.0832],
    [(0.0096 + 0.0448) / 0.0832, (0.0096 + 0.0448) / 0.0832]
])

# Validate results
assert np.isclose(forward_log_prob, expected_log_prob, rtol=TOLERANCE, atol=0)
assert np.isclose(backward_log_prob, expected_log_prob, rtol=TOLERANCE, atol=0)
assert np.allclose(result["posterior"], expected_posterior, rtol=TOLERANCE, atol=0)

print("Test 2 passed!")

Test 2 passed!


In [6]:
# Test 3: Provided observation sequence
observations = "ATGCAA"

forward_matrix, forward_log_prob = forward(hmm, observations)
backward_matrix, backward_log_prob = backward(hmm, observations)
result = forward_backward(hmm, observations)

# Validate results
assert np.isclose(forward_log_prob, backward_log_prob, rtol=TOLERANCE, atol=0)
assert np.allclose(result["posterior"].sum(axis=0), 1, rtol=TOLERANCE, atol=0)
assert result["posterior"].shape == (2, 6)

print("Forward log-probability:", forward_log_prob)
print("Backward log-probability:", backward_log_prob)
print("Posterior probabilities:\n", result["posterior"])
print("Test 3 passed!")

Forward log-probability: -8.623461496768254
Backward log-probability: -8.623461496768252
Posterior probabilities:
 [[0.8639135  0.8359201  0.58060576 0.58060576 0.8359201  0.8639135 ]
 [0.1360865  0.1640799  0.41939424 0.41939424 0.1640799  0.1360865 ]]
Test 3 passed!


In [7]:
# Test 4: Three-state model
init_probs_three = {"S1": 0.5, "S2": 0.3, "S3": 0.2}

trans_probs_three = {
    "S1": {"S1": 0.6, "S2": 0.3, "S3": 0.1},
    "S2": {"S1": 0.2, "S2": 0.5, "S3": 0.3},
    "S3": {"S1": 0.1, "S2": 0.3, "S3": 0.6}
}

emit_probs_three = {
    "S1": {"x": 0.8, "y": 0.2},
    "S2": {"x": 0.5, "y": 0.5},
    "S3": {"x": 0.2, "y": 0.8}
}

hmm_three = HMM(init_probs_three, trans_probs_three, emit_probs_three)
observations = "xyxy"

forward_matrix, forward_log_prob = forward(hmm_three, observations)
backward_matrix, backward_log_prob = backward(hmm_three, observations)
result = forward_backward(hmm_three, observations)

# Validate results
assert np.isclose(forward_log_prob, backward_log_prob, rtol=TOLERANCE, atol=0)
assert np.allclose(result["posterior"].sum(axis=0), 1, rtol=TOLERANCE, atol=0)
assert result["posterior"].shape == (3, 4)

print("Test 4 passed!")

Test 4 passed!


In [8]:
# Test 5: Viterbi vs. posterior decoding
observations = "ACC"

viterbi_path, viterbi_log_prob = hmm.viterbi_algorithm(observations)
result = forward_backward(hmm, observations)

# Select the most probable state at each position
posterior_indices = np.argmax(result["posterior"], axis=0)
posterior_path = [hmm.states[i] for i in posterior_indices]

# Validate results
assert list(viterbi_path) == ["E", "E", "E"]
assert posterior_path == ["E", "I", "I"]

print("Viterbi path:", viterbi_path)
print("Posterior path:", posterior_path)
print("Test 5 passed!")

Viterbi path: ['E', 'E', 'E']
Posterior path: ['E', 'I', 'I']
Test 5 passed!


In [9]:
# Test 6: Zero probabilities
init_probs_zero = {"S1": 1.0, "S2": 0.0}

trans_probs_zero = {
    "S1": {"S1": 0.5, "S2": 0.5},
    "S2": {"S1": 0.0, "S2": 1.0}
}

emit_probs_zero = {
    "S1": {"x": 1.0, "y": 0.0},
    "S2": {"x": 0.0, "y": 1.0}
}

hmm_zero = HMM(init_probs_zero, trans_probs_zero, emit_probs_zero)

result = forward_backward(hmm_zero, "xy")

# Expected values
expected_log_prob = np.log(0.5)
expected_posterior = np.array([[1.0, 0.0], [0.0, 1.0]])

# Validate results
assert np.isclose(result["log_prob"], expected_log_prob, rtol=TOLERANCE, atol=0)
assert np.allclose(result["posterior"], expected_posterior, rtol=TOLERANCE, atol=0)

print("Test 6 passed!")

Test 6 passed!


In [10]:
# Test 7: Invalid observations
for algorithm in [forward, backward, forward_backward]:
    for observations in ["", "ACN"]:
        try:
            algorithm(hmm, observations)
        except ValueError:
            pass
        else:
            raise AssertionError("Invalid observation was accepted")

print("Test 7 passed!")

Test 7 passed!


In [11]:
# Test 8: Long observation sequence
observations = "ATGCAA" * 1000

forward_matrix, forward_log_prob = forward(hmm, observations)
backward_matrix, backward_log_prob = backward(hmm, observations)
result = forward_backward(hmm, observations)

# Validate numerical stability
assert np.isfinite(forward_log_prob)
assert np.isfinite(backward_log_prob)
assert np.isclose(forward_log_prob, backward_log_prob, rtol=TOLERANCE, atol=0)
assert np.allclose(result["posterior"].sum(axis=0), 1, rtol=TOLERANCE, atol=0)

print("Sequence length:", len(observations))
print("Forward log-probability:", forward_log_prob)
print("Backward log-probability:", backward_log_prob)
print("Test 8 passed!")

Sequence length: 6000
Forward log-probability: -8487.823689332818
Backward log-probability: -8487.823689333723
Test 8 passed!
